In [ ]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
ENGLISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_kaggle.csv')
SPANISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle.csv')

BATCH_SIZE = 100
NAMES = ['label', 'text']
MODEL_NAME = 'Helsinki-NLP/opus-mt-en-es'

print("Loading dataset...")
dataset = pd.DataFrame()
try:
    dataset = pd.read_csv(ENGLISH_FILE)
    dataset.rename(columns={dataset.columns[0]: NAMES[0], dataset.columns[1]: NAMES[1]}, inplace=True)
    dataset = dataset[NAMES]
    print(f"Dataset loaded successfully. Total rows: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

print(f"Loading model {MODEL_NAME}...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

def translate_batch(texts, tokenizer, model, device):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

print("Starting translation...")

spanish_texts = []
total_texts = len(dataset)

for i in tqdm(range(0, total_texts, BATCH_SIZE)):
    batch_texts = dataset['text'].iloc[i:i + BATCH_SIZE].tolist()
    translated_texts = translate_batch(batch_texts, tokenizer, model, device)
    spanish_texts.extend(translated_texts)

dataset['text_es'] = spanish_texts
print("Example translations:")
print(dataset[['text', 'text_es']].head())

dataset_final = dataset[['label', 'text_es']].rename(columns={'text_es': 'text'})
dataset_final.to_csv(SPANISH_FILE, index=False)

print(f"Translation completed. Translated dataset saved to {SPANISH_FILE}.")